# Mission 3 학습 실험 로그

이 노트북은 위에서부터 차례로 실행합니다. 학습에는 서울 Training JSON만 사용하고, 모델 입력으로는 `utterances[].text`만 사용합니다.

실행이 끝나면 출력 로그가 사라지지 않도록 노트북을 저장합니다.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "src" / "mission3").is_dir():
    repo_root = cwd
elif cwd.name == "mission3" and cwd.parent.name == "src":
    repo_root = cwd.parents[1]
else:
    raise RuntimeError("DCC_AIdeas 또는 src/mission3 폴더에서 노트북을 실행하세요.")

mission_dir = repo_root / "src" / "mission3"
sys.path.insert(0, str(mission_dir))
train_label_dir = repo_root / "DATA" / "대학부 데이터" / "Training" / "2.라벨링데이터" / "TL_서울_구급"
checkpoint_path = repo_root / "saved_models" / "mission3_model.ckpt"

print("repo_root:", repo_root)
print("Training labels exist:", train_label_dir.is_dir())

## 1. Training 데이터 확인

아래 셀은 파일 개수, 빈 텍스트 개수, 9개 증상의 정답 개수를 확인합니다. 아직 모델을 학습하지는 않습니다.

In [ ]:
from collections import Counter
from common import TARGET_SYMPTOMS, dialogue_text, find_label_files, load_annotation, target_symptoms

label_files = find_label_files(train_label_dir)
symptom_counts = Counter()
empty_text_count = 0

for path in label_files:
    annotation = load_annotation(path)
    empty_text_count += not bool(dialogue_text(annotation).strip())
    symptom_counts.update(target_symptoms(annotation))

print("Training JSON:", len(label_files))
print("빈 대화 텍스트:", empty_text_count)
for symptom in TARGET_SYMPTOMS:
    print(f"{symptom:<8} {symptom_counts[symptom]:>6}")

## 2. 베이스라인 학습

아래 셀은 Training 데이터 내부 80/20 분할로 Macro F1을 확인한 후, Training 전체로 최종 모델을 저장합니다. Validation 폴더는 읽지 않습니다.

In [ ]:
import subprocess

command = [
    sys.executable,
    str(mission_dir / "train.py"),
    "--label_dir", str(train_label_dir),
    "--ckpt_path", str(checkpoint_path),
]
print(" ".join(command))
subprocess.run(command, cwd=repo_root, check=True)

## 3. 실험 메모

- 실험 날짜:
- 모델: char TF-IDF + One-vs-Rest Logistic Regression
- 주요 설정: `ngram_range=(2, 5)`, `max_features=100000`, `C=2.0`
- Training 내부 Macro F1:
- 가장 잘 맞힌 증상:
- 가장 못 맞힌 증상:
- 다음에 하나만 바꿀 설정:

> 위 메모를 채우고 모든 셀의 출력이 보이는 상태로 노트북을 저장하세요.